[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/125/blob/main/06_series/06_4_Text_in_a_Series.ipynb)

# 06.4: Text in a Series

You already know string methods such as `.lower()` and `.split()` from module 05, and they work on one string at a time. A series can hold thousands of strings, and you do not want to write a loop every time you need to search or clean them. This notebook introduces the `.str` accessor, which applies a string method to every value in a series.

This notebook opens in Google Colab. If Colab warns that the notebook was not authored by Google, click **Run anyway**. Your changes are not kept when you close the tab unless you choose **File > Save a copy in Drive**, so do that if you want to keep your work.

## The data

`names` is a series holding the major names. Its index is also the major names, so that a mask made from `names` lines up with `salaries`. Later in the notebook we also use `books`, a series of 57 best-selling books, each written as a title and an author.

In [ ]:
import pandas as pd

majors = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_recent_grads.csv")
major_list = majors["Major"].to_list()
salaries = pd.Series(majors["Median"].to_list(), index=major_list)
names = pd.Series(major_list, index=major_list)

books = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_book_bestsellers.csv")["0"].rename("book")
books.head(3)

Each value is one string, such as `The Little Prince by Antoine de Saint-Exupéry`.

## The `.str` accessor

A series does not have a `.lower()` method of its own, because `.lower()` is something that strings do. The next cell fails on purpose. If you use Run all, Colab stops at this cell, so run the cells after it yourself.

In [ ]:
names.lower()

You could write a loop that goes through the strings one at a time, but that is more code and it runs more slowly. Putting `.str` before the method tells pandas to use the string method on each value. The result is a new series with the same labels. Most string methods you know have a `.str` version, including `.upper()`.

In [ ]:
names.str.lower().head(3)

The values are now lower case, for example `petroleum engineering`, and the labels did not change. `.str.len()` gives the length of each string, which is how you can find the longest name.

In [ ]:
names.str.len().sort_values().tail(2)

The longest name has 65 characters: ELECTRICAL, MECHANICAL, AND PRECISION TECHNOLOGIES AND PRODUCTION. A mask built from `.str.len()` selects values by their length, which finds the shortest book in the list.

In [ ]:
book_lengths = books.str.len()
print(book_lengths.min())
books[book_lengths == book_lengths.min()]

The shortest entry has 20 characters: Cosmos by Carl Sagan. Finding the minimum first and then using it in the mask means that the code still works if the data changes.

## Searching: `.str.contains()`

`.str.contains()` makes a boolean mask that is `True` for every value that includes a piece of text. Everything you learned about masks in 06.2 applies, so you can count with it and filter with it.

In [ ]:
print(names.str.contains("ENGINEERING").sum())
names[names.str.contains("PSYCHOLOGY")]

Twenty-six major names include ENGINEERING. The list for PSYCHOLOGY has 8 majors, and one of them is COGNITIVE SCIENCE AND BIOPSYCHOLOGY, which matches because `.str.contains()` looks for the text anywhere, including inside a longer word. Matching is also case-sensitive, so searching for lower case text finds nothing in this data, and pandas gives no warning about it.

In [ ]:
names.str.contains("engineering").sum()

A result of 0 could mean that no values match, or it could mean a problem with your search, so check any count that looks wrong. One other detail: `.str.contains()` treats a few characters as special by default, including the period, which matches any character. If your search text includes punctuation, add `regex=False` to search for exactly what you typed. The two counts below differ for that reason.

In [ ]:
print(books.str.contains(".").sum())
print(books.str.contains(".", regex=False).sum())

The first search counts all 57 values, because every value has at least one character. The second counts only the 15 values with an actual period. If a series has missing values, `.str.contains()` gives a missing result for those rows, and using it as a mask raises an error. Adding `na=False` treats those rows as not matching. There are no missing values here.

A mask from text is useful for comparing groups. Since `names` and `salaries` have the same index, the mask can select from `salaries`.

In [ ]:
is_engineering = names.str.contains("ENGINEERING")
print(salaries[is_engineering].median())
print(salaries[~is_engineering].median())

The median across majors with ENGINEERING in the name is 57050, and across all other majors it is 35000.

## Starts with, ends with and find

`.str.startswith()` and `.str.endswith()` also produce masks, and they are stricter than `.str.contains()` because the text has to be in a specific place.

In [ ]:
print(books.str.endswith("Tolkien").sum())
names[names.str.startswith("GENERAL")]

One book ends with Tolkien (The Hobbit). Six majors start with GENERAL, including GENERAL ENGINEERING and GENERAL BUSINESS, and a name with GENERAL only in the middle would not match. `.str.find()` gives the position where the text first appears in each string, or -1 if it does not appear, just like the string method `.find()`.

In [ ]:
books.str.find("Rowling").head(3)

The first book has Rowling starting at position 50, and the next two do not have it at all, so they show -1.

## Replacing text

`.str.replace()` swaps one piece of text for another in every value. As with the operations in 06.2, the result is a new series unless you assign it.

In [ ]:
names.str.replace(" AND ", " & ").head(4)

Where a name had AND between spaces, such as MINING AND MINERAL ENGINEERING, it now has an ampersand. Names without it are unchanged.

## Splitting text

`.str.split()` breaks each string into a list of pieces, and `.str[0]` then takes the first piece of each list. The `.str` is needed again because each value is now a list. `.str[-1]` takes the last piece, using the same negative indexing as a list.

In [ ]:
first_word = names.str.split(" ").str[0]
first_word.value_counts().head(3)

The most common first word is MISCELLANEOUS with 10 majors, followed by COMPUTER and GENERAL with 6 each. Splitting, taking a piece and then calling `.value_counts()` is a very common way to pull information out of text. Each book is written as a title, then ` by `, then an author, so the same steps give us the authors.

In [ ]:
authors = books.str.split(" by ").str[1]
authors.value_counts().head(2)

J. K. Rowling has 7 of the books and Dan Brown has 3. When you split like this, check that each value breaks into the pieces you expect, because a split on text that appears twice gives more pieces than you planned for. `.str.len()` on a series of lists gives the length of each list.

In [ ]:
pieces = books.str.split(" by ")
books[pieces.str.len() != 2]

One book has an extra ` by `: Charlotte's Web by E. B. White; illustrated by Garth Williams. Python gave no warning, and for this book `.str[1]` is `E. B. White; illustrated`.

## Cleaning up text

Text typed by people often has stray spaces and inconsistent capitals. To Python, each of these values is a different string, so they would be counted separately.

In [ ]:
answers = pd.Series(["Yes", "yes ", " YES", "No", "no"])
answers.value_counts()

Each of the five answers is counted as its own category, even though there are really two answers. `.str.strip()` removes spaces and other whitespace from the ends of each string. To apply two methods in a row, chain them, with `.str` before each one. Chaining is convenient, but a long chain is hard to read and to fix, so split it into steps with names when it gets long.

In [ ]:
answers.str.strip().str.lower().value_counts()

Now `yes` has a count of 3 and `no` has a count of 2.

## Practice problems

These problems are not collected or graded. They are here so that you can find out whether you can write this code yourself before the reading check and the pair activity. Write your answer in the cell under each problem, run it, and then run the check cell below it, which tells you whether your answer is right. The check cells are collapsed because they contain the answers, so try the problem before you open one.

Run the next two cells first. The first one creates fresh data for the problems, and the second sets up the checks.

In [ ]:
import pandas as pd

majors = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_recent_grads.csv")
major_list = majors["Major"].to_list()
salaries = pd.Series(majors["Median"].to_list(), index=major_list)
names = pd.Series(major_list, index=major_list)
books = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_book_bestsellers.csv")["0"].rename("book")
responses = pd.Series([" Red", "red", "BLUE ", "blue", " Red "])

In [ ]:
#@title Run this cell to set up the checks
import numpy as np
import pandas as pd


def same(got, want):
    """True when got matches want: same labels and values for a series, same value for a number."""
    try:
        if isinstance(want, pd.Series):
            if not isinstance(got, pd.Series) or list(got.index) != list(want.index):
                return False
            if pd.api.types.is_numeric_dtype(want):
                return bool(np.allclose(got.astype(float), want.astype(float), equal_nan=True))
            return list(got) == list(want)
        if isinstance(want, (list, np.ndarray)):
            return list(got) == list(want)
        if isinstance(want, str):
            return got == want
        if isinstance(got, (pd.Series, pd.DataFrame, list, np.ndarray)):
            return False
        return bool(np.isclose(float(got), float(want)))
    except (TypeError, ValueError):
        return False


def check(number, name, want, hint, unchanged=None):
    """Print feedback for one problem. unchanged is (variable name, its starting value)."""
    if name not in globals():
        print(f"Problem {number}: not yet. No variable named {name} exists. Did you run your answer cell?")
    elif not same(globals()[name], want):
        print(f"Problem {number}: not yet. {hint}")
    elif unchanged is not None and not globals()[unchanged[0]].equals(unchanged[1]):
        print(f"Problem {number}: {name} is right, but {unchanged[0]} changed. Work on a copy, then run the setup cell again.")
    else:
        print(f"Problem {number}: correct.")

### Problem 1
Create a series named `upper_books` that has every value of `books` in upper case.

In [ ]:
# Problem 1

In [ ]:
#@title Check Problem 1
check(1, "upper_books", books.str.upper(), "Use the .str accessor with .upper().")

### Problem 2
Store the number of major names that contain the text `SCIENCE` in a variable named `science_count`.

In [ ]:
# Problem 2

In [ ]:
#@title Check Problem 2
check(2, "science_count", names.str.contains("SCIENCE").sum(), "Make a mask with .str.contains() and add it up.")

### Problem 3
Create a series named `computer_majors` that has only the items of `names` that start with `COMPUTER`.

In [ ]:
# Problem 3

In [ ]:
#@title Check Problem 3
check(3, "computer_majors", names[names.str.startswith("COMPUTER")], "Use .str.startswith() as a mask inside the square brackets.")

### Problem 4
Store the length of the longest major name in a variable named `longest`.

In [ ]:
# Problem 4

In [ ]:
#@title Check Problem 4
check(4, "longest", names.str.len().max(), "Get the length of each string and then take the largest.")

### Problem 5
Each value in `books` has the title before ` by `. Create a series named `titles` that holds only the titles.

In [ ]:
# Problem 5

In [ ]:
#@title Check Problem 5
check(5, "titles", books.str.split(" by ").str[0], "Split on ' by ' with the spaces and take the first piece.")

### Problem 6
Store the median of `salaries` across the majors whose names contain `EDUCATION` in a variable named `education_median`.

In [ ]:
# Problem 6

In [ ]:
#@title Check Problem 6
check(6, "education_median", salaries[names.str.contains("EDUCATION")].median(), "Build the mask from names and use it to select from salaries.")

### Problem 7
`responses` has answers with messy spaces and capitals. Create a series named `clean_counts` that gives the counts after removing the extra spaces and making everything lower case.

In [ ]:
# Problem 7

In [ ]:
#@title Check Problem 7
check(7, "clean_counts", responses.str.strip().str.lower().value_counts(), "Chain .str.strip() and .str.lower(), and then use .value_counts().")

## What's next

You can now search, change and split the text in a series. Sometimes the labels themselves define groups, as when each row is a championship and the label is the team. Notebook 06.5 shows how to get a summary for each group.